# Dify 电商客服评测 v4.0

适配 eval_set_v4.json（505条用例）

**核心改进：**
1. 从 JSON 文件加载评测集
2. 支持多轮对话（conversation_context）
3. 禁止表述检测（must_not 字段）
4. 断点续评（每10条自动保存）
5. 增强裁判 prompt（覆盖全维度）
6. 详细数据分析报告 + CSV 导出

In [1]:
!pip install openai requests pandas -q

## Cell 1: 配置

In [2]:
import os
import sys
import json
import time
import requests
from getpass import getpass
from typing import Optional, List, Dict
from openai import OpenAI
import pandas as pd
from datetime import datetime
from collections import Counter

print("=== Dify 电商客服评测 v4.0 - 配置 ===")

# Dify 应用配置
DIFY_API_URL = input("Dify 应用 API URL（默认 http://localhost/v1/chat-messages）: ").strip()
if not DIFY_API_URL:
    DIFY_API_URL = "http://localhost/v1/chat-messages"
DIFY_API_KEY = getpass("Dify 应用 API Key: ")

# 阿里云百炼（裁判模型）
ALIYUN_API_KEY = getpass("阿里云百炼 API Key: ")
judge_client = OpenAI(
    api_key=ALIYUN_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
JUDGE_MODEL = "qwen-plus"

# 评测集路径
EVAL_SET_PATH = input("\n评测集文件路径（默认 eval_set_v4.json）: ").strip()
if not EVAL_SET_PATH:
    EVAL_SET_PATH = "eval_set_v4.json"

# 输出目录
OUTPUT_DIR = input("结果输出目录（默认 eval_output）: ").strip()
if not OUTPUT_DIR:
    OUTPUT_DIR = "eval_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


print(f"\n✅ 配置完成")
print(f"   Dify URL: {DIFY_API_URL}")
print(f"   裁判模型: {JUDGE_MODEL}（阿里云百炼）")
print(f"   评测集: {EVAL_SET_PATH}")
print(f"   输出目录: {OUTPUT_DIR}")

=== Dify 电商客服评测 v4.0 - 配置 ===


Dify 应用 API URL（默认 http://localhost/v1/chat-messages）:  
Dify 应用 API Key:  ········
阿里云百炼 API Key:  ········

评测集文件路径（默认 eval_set_v4.json）:  
结果输出目录（默认 eval_output）:  



✅ 配置完成
   Dify URL: http://localhost/v1/chat-messages
   裁判模型: qwen-plus（阿里云百炼）
   评测集: eval_set_v4.json
   输出目录: eval_output


## Cell 2: 加载评测集

In [3]:
with open(EVAL_SET_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

eval_cases = eval_data['cases']
meta = eval_data.get('meta', {})

print(f"✅ 评测集加载完成")
print(f"   版本: {meta.get('version', '?')}")
print(f"   总用例数: {len(eval_cases)}")

type_dist = Counter(c['test_type'] for c in eval_cases)
cat_dist = Counter(c['category'] for c in eval_cases)
print(f"\n   按类型: {dict(type_dist)}")
print(f"   按分支: {dict(cat_dist)}")

✅ 评测集加载完成
   版本: 4.0
   总用例数: 505

   按类型: {'single_turn': 382, 'multi_turn': 17, 'boundary': 35, 'prohibited': 23, 'adversarial': 48}
   按分支: {'商品咨询': 281, '订单查询': 36, '退款换货': 70, '投诉': 47, '其他': 49, '边界混淆': 22}


## Cell 3: 核心函数定义

包含：调用Dify、多轮对话、裁判评分、断点续评

In [4]:
# ========== 调用 Dify（单轮）==========
def call_dify(user_query: str, conversation_id: Optional[str] = None):
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "inputs": {},
        "query": user_query,
        "user": "eval-user",
        "response_mode": "blocking"
    }
    if conversation_id:
        data["conversation_id"] = conversation_id
    try:
        resp = requests.post(DIFY_API_URL, headers=headers, json=data, timeout=60)
        resp.raise_for_status()
        result = resp.json()
        return result.get("answer", ""), result.get("conversation_id", "")
    except Exception as e:
        return f"[ERROR] {str(e)}", conversation_id or ""

# ========== 调用 Dify（多轮）==========
def call_dify_multi_turn(case: dict):
    context = case.get('conversation_context', [])
    if not context:
        return call_dify(case['user_query'])
    conversation_id = None
    for turn in context:
        if turn['role'] == 'user':
            answer, conversation_id = call_dify(turn['content'], conversation_id)
            time.sleep(0.3)
    final_answer, final_conv_id = call_dify(case['user_query'], conversation_id)
    return final_answer, final_conv_id

# ========== 确定性预检：must_not 精确子串匹配 ==========
def check_must_not(response: str, must_not_list: list) -> bool:
    """精确子串匹配，返回 True 表示违规"""
    if not must_not_list:
        return False
    for item in must_not_list:
        item = item.strip()
        if not item:
            continue
        if item in response:
            return True
    return False

# ========== 确定性预检：expected_points 宽松匹配 ==========
def check_expected_points(response: str, expected_points: list) -> bool:
    """宽松匹配：关键词或其变形出现在回复中即算命中，全部命中返回 True"""
    if not expected_points:
        return True
    resp_lower = response.lower()
    for point in expected_points:
        point = point.strip()
        if not point:
            continue
        # 直接子串匹配
        if point in response:
            continue
        # 宽松匹配：去掉标点后再查
        point_clean = point.replace("，", "").replace("。", "").replace("、", "")
        if point_clean in response:
            continue
        # 部分关键词匹配（取前2-4个字符）
        if len(point) >= 2:
            short = point[:min(4, len(point))]
            if short in response:
                continue
        # 未命中
        return False
    return True

# ========== 裁判评分（v4.1 修复版）==========
def llm_judge(case: dict, actual_response: str) -> dict:
    must_not_str = "、".join(case.get('must_not', [])) if case.get('must_not') else "无"
    expected_points = "、".join(case.get('expected_answer_contains', [])) if case.get('expected_answer_contains') else "无特殊要求"
    context_str = ""
    if case.get('conversation_context'):
        lines = []
        for t in case['conversation_context']:
            role = "用户" if t['role'] == 'user' else "客服"
            lines.append(f"{role}: {t['content'][:80]}")
        context_str = "\n".join(lines)
    complaint_level = case.get('expected_complaint_level') or "不适用"

    prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

## 评分规则（必须严格遵守）

### branch_routing（分支路由）评分规则
- 你无法看到系统的实际路由 trace，请根据回复内容体现的处理逻辑判断
- 5分：回复内容与期望分支"{case['expected_branch']}"的处理逻辑一致，体现了该分支应有的行为特征
- 3-4分：回复基本符合期望分支，但有少量偏差
- 0-1分：回复内容明显属于其他分支（如期望退款但回复了商品参数）
- 重要：不要因为回复中提到了其他分支的术语（如退款分支中提到"7天无理由退货"）就判定路由错误——各分支的回复中合理提及其他业务规则是正常的
- 重要：只要回复体现了期望分支的核心动作（如退款分支体现退款处理、订单查询体现查询结果），即应给4分以上

### must_not_violation（禁止表述违规）判定规则
- 逐条检查 must_not 列表，使用精确子串匹配
- 只有当回复中完整包含 must_not 中的某个表述时，才判定为违规
- 禁止使用语义近似、同义词扩展来判断
- 否定表述不算违规：如 must_not=["可办理退货"]，回复中有"不可办理退货"→不违规（语义相反）
- 不同表述不算违规：如 must_not=["7天内可退"]，回复中有"7天无理由退货"→不违规（不同表述）

### contains_expected_points（要点覆盖）判定规则
- 逐条检查 expected_answer_contains 中的每个要点
- 使用宽松匹配：关键词的变形、包含该关键词的 longer phrase 均算命中
  如"未签收"匹配"尚未签收"、"运输中"匹配"目前运输中"
- 只要回复中体现了该要点的核心语义，即算命中
- 全部命中→true，有任何一个未命中→false

### boundary_correctness（边界正确性）评分规则
- 根据回复是否正确处理了边界值说明中的业务逻辑来评分
- 只要回复体现了边界值对应的核心处理逻辑，即给4分以上
- 不要因为回复提供了额外的替代方案就扣分

## 用例信息
【用例ID】{case['case_id']}
【测试类型】{case['test_type']}
【用户问题（最后一轮）】{case['user_query']}
【对话上下文】
{context_str if context_str else '无（单轮对话）'}

【期望意图】{case['expected_intent']}
【期望分支】{case['expected_branch']}
【期望IF/ELSE路径】{', '.join(case.get('expected_if_else_path', []))}
【期望答案应包含的关键要点】{expected_points}
【回复中禁止出现的表述】{must_not_str}
【期望工具调用】{', '.join(case.get('expected_tool_calls', [])) if case.get('expected_tool_calls') else '无'}
【期望投诉分级】{complaint_level}
【是否期望转人工】{case.get('expected_human_handoff', False)}
【边界值说明】{case.get('boundary_value') or '无'}

【实际客服回复】
{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "must_not_violation": <true/false>,
    "boundary_correctness": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "handoff_correctness": <0-5, 不适用填-1>,
    "comment": "<80字以内>"
}}

只输出 JSON。"""

    # 第一优先：带 response_format 调用
    try:
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        score = json.loads(resp.choices[0].message.content)
    except Exception as e1:
        # 第二优先：去掉 response_format 重试
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            content = resp.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1] if "\n" in content else content[3:]
            if content.endswith("```"):
                content = content.rsplit("```", 1)[0]
            content = content.strip()
            if content.startswith("json"):
                content = content[4:].strip()
            score = json.loads(content)
        except Exception as e2:
            score = {
                "error": str(e1), "intent_accuracy": 0, "branch_routing": 0,
                "answer_relevance": 0, "information_completeness": 0,
                "faithfulness": 0, "tone_compliance": 0,
                "must_not_violation": False, "boundary_correctness": 0,
                "is_hallucination": False, "contains_expected_points": False,
                "handoff_correctness": -1, "comment": f"裁判调用失败: {e1}"
            }

    # ========== 确定性预检覆盖裁判误判 ==========
    must_not_list = case.get('must_not', [])
    expected_points_list = case.get('expected_answer_contains', [])

    # must_not：用代码精确匹配覆盖裁判的语义误判
    code_must_not = check_must_not(actual_response, must_not_list)
    if not code_must_not and score.get('must_not_violation', False):
        # 代码没检出但裁判说违规 → 用代码结果覆盖
        score['must_not_violation'] = False
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]must_not未违规(精确匹配)。{orig_comment}"

    # expected_points：用代码宽松匹配覆盖裁判的漏判
    code_points = check_expected_points(actual_response, expected_points_list)
    if code_points and not score.get('contains_expected_points', False):
        # 代码判定命中但裁判说没命中 → 用代码结果覆盖
        score['contains_expected_points'] = True
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]要点已覆盖(宽松匹配)。{orig_comment}"

    return score

# ========== 单条执行 ==========
def evaluate_single_case(case: dict) -> dict:
    actual_response, conv_id = call_dify_multi_turn(case)
    score = llm_judge(case, actual_response)
    return {
        "case_id": case["case_id"], "test_type": case["test_type"],
        "category": case["category"], "difficulty": case["difficulty"],
        "user_query": case["user_query"],
        "expected_intent": case["expected_intent"],
        "expected_branch": case["expected_branch"],
        "expected_if_else_path": ",".join(case.get("expected_if_else_path", [])),
        "expected_answer_contains": ",".join(case.get("expected_answer_contains", [])),
        "must_not": ",".join(case.get("must_not", [])),
        "expected_complaint_level": case.get("expected_complaint_level"),
        "expected_human_handoff": case.get("expected_human_handoff", False),
        "expected_retrieval_hit": case.get("expected_retrieval_hit"),
        "boundary_value": case.get("boundary_value"),
        "is_multi_turn": len(case.get("conversation_context", [])) > 0,
        "conversation_id": conv_id,
        "actual_response": actual_response,
        "score": score
    }

# ========== 断点续评 ==========
def load_checkpoint():
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def save_checkpoint(results):
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ 函数定义完成（v4.1 修复版：含确定性预检 + 增强裁判 prompt）")

✅ 函数定义完成（v4.1 修复版：含确定性预检 + 增强裁判 prompt）


## Cell 4: 连通性测试

In [21]:
print("=== 连通性测试（第1条用例）===\n")
test_case = eval_cases[0]
result = evaluate_single_case(test_case)
print(f"用例: {test_case['user_query']}")
print(f"回复: {result['actual_response'][:200]}...")
print(f"\n评分:")
print(json.dumps(result['score'], ensure_ascii=False, indent=2))
print("\n✅ 连通性正常！继续运行下一个 Cell 批量评测")

=== 连通性测试（第1条用例）===

用例: 索尼WF-1000XM5蓝牙耳机的降噪效果怎么样？
回复: **索尼WF-1000XM5蓝牙耳机的降噪深度可达42dB，能有效隔绝外界噪音。**这款耳机特别适合通勤和学习使用。关于续航方面，其续航时间为30小时，能满足日常及长途旅行的需求。如果您有更多问题，请随时告知！...

评分:
{
  "intent_accuracy": 5,
  "branch_routing": 5,
  "answer_relevance": 5,
  "information_completeness": 5,
  "faithfulness": 5,
  "tone_compliance": 5,
  "must_not_violation": false,
  "boundary_correctness": 5,
  "is_hallucination": false,
  "contains_expected_points": true,
  "handoff_correctness": 5,
  "comment": "回复精准匹配咨询意图（consult）和商品咨询分支；紧扣用户问题聚焦降噪效果，明确包含关键要点'降噪'和'42dB'；信息准确、无冗余无关内容（如续航属额外补充，但未偏离主题且未违反禁止项，属合理增值信息）；语气专业友好；未出现禁止表述；无需工具调用与转人工，执行正确；无事实性错误或幻觉。"
}

✅ 连通性正常！继续运行下一个 Cell 批量评测


## Cell 5: 批量评测

支持断点续评，每10条自动保存。可修改 `START_INDEX` 和 `BATCH_SIZE` 控制评测范围。

In [8]:
# ===== 可调参数 =====
START_INDEX = 0      # 从第几条开始（0-based）
BATCH_SIZE = 505      # 评测多少条（0=全部505条）
# ====================

# 加载断点
results = load_checkpoint()
completed_ids = set(r["case_id"] for r in results)

# 确定范围
if BATCH_SIZE > 0:
    eval_range = eval_cases[START_INDEX:START_INDEX + BATCH_SIZE]
else:
    eval_range = eval_cases[START_INDEX:]

pending = [c for c in eval_range if c['case_id'] not in completed_ids]
total = len(pending)

print(f"=== 批量评测 ===")
print(f"  总用例: {len(eval_cases)}")
print(f"  已完成: {len(results)}（断点续评）")
print(f"  本次待评: {total}")
if total > 0:
    print(f"  预估耗时: ~{total * 3 // 60}分{total * 3 % 60}秒")
print()

if total == 0:
    print("✅ 所有用例已完成")
else:
    start_time = time.time()
    for i, case in enumerate(pending, 1):
        elapsed = time.time() - start_time
        eta = (elapsed / (i-1) * (total - i + 1)) if i > 1 else total * 3
        print(f"[{i}/{total}] ({elapsed:.0f}s/{eta:.0f}s) {case['case_id']}...", end="", flush=True)
        try:
            r = evaluate_single_case(case)
            results.append(r)
            s = r['score']
            bs = s.get('branch_routing', 0)
            hal = s.get('is_hallucination', False)
            vio = s.get('must_not_violation', False)
            status = "✅" if bs >= 4 else "⚠️"
            flags = []
            if hal: flags.append("幻觉")
            if vio: flags.append("违规")
            fl = f"[{','.join(flags)}]" if flags else ""
            print(f" {status} 分支={bs}/5{fl}")
        except Exception as e:
            print(f" ❌ {e}")
            results.append({"case_id": case["case_id"], "category": case["category"],
                "user_query": case["user_query"], "expected_branch": case["expected_branch"],
                "actual_response": f"[ERROR] {e}", 
                "score": {"branch_routing": 0, "comment": str(e)}})
        if i % 10 == 0:
            save_checkpoint(results)
            print(f"  💾 已保存（{len(results)}条）")
        time.sleep(0.5)
    save_checkpoint(results)
    t = time.time() - start_time
    print(f"\n✅ 完成！共{len(results)}条，耗时{t//60:.0f}分{t%60:.0f}秒")

=== 批量评测 ===
  总用例: 505
  已完成: 400（断点续评）
  本次待评: 105
  预估耗时: ~5分15秒

[1/105] (0s/315s) TC-CMP-L1-003... ⚠️ 分支=1/5
[2/105] (18s/1886s) TC-CMP-L1-004... ✅ 分支=5/5
[3/105] (35s/1822s) TC-CMP-L1-005... ✅ 分支=5/5
[4/105] (52s/1783s) TC-CMP-L1-006... ⚠️ 分支=3/5
[5/105] (64s/1604s) TC-CMP-L1-007... ⚠️ 分支=2/5
[6/105] (73s/1453s) TC-CMP-L1-008... ⚠️ 分支=1/5[幻觉]
[7/105] (90s/1489s) TC-CMP-L1-009... ⚠️ 分支=1/5
[8/105] (98s/1371s) TC-CMP-L1-010... ✅ 分支=5/5
[9/105] (115s/1393s) TC-CMP-L1-011... ⚠️ 分支=1/5
[10/105] (132s/1408s) TC-CMP-L1-012... ⚠️ 分支=1/5
  💾 已保存（410条）
[11/105] (144s/1366s) TC-CMP-L2-001... ✅ 分支=5/5
[12/105] (159s/1363s) TC-CMP-L2-002... ✅ 分支=5/5
[13/105] (176s/1366s) TC-CMP-L2-003... ✅ 分支=5/5
[14/105] (192s/1361s) TC-CMP-L2-004... ✅ 分支=5/5
[15/105] (210s/1362s) TC-CMP-L2-005... ✅ 分支=5/5
[16/105] (227s/1362s) TC-CMP-L2-006... ⚠️ 分支=1/5[幻觉]
[17/105] (245s/1364s) TC-CMP-L2-007... ✅ 分支=5/5
[18/105] (262s/1354s) TC-CMP-L2-008... ✅ 分支=5/5
[19/105] (279s/1346s) TC-CMP-L3-001... ✅ 分支=5/5
[20/105]

## Cell 6: 生成报告 + 导出

In [9]:
total = len(results)
rows = []
for r in results:
    s = r.get('score', {})
    rows.append({
        'case_id': r['case_id'],
        'test_type': r.get('test_type', ''),
        'category': r.get('category', ''),
        'difficulty': r.get('difficulty', ''),
        'is_multi_turn': r.get('is_multi_turn', False),
        'intent_accuracy': s.get('intent_accuracy', 0),
        'branch_routing': s.get('branch_routing', 0),
        'answer_relevance': s.get('answer_relevance', 0),
        'information_completeness': s.get('information_completeness', 0),
        'faithfulness': s.get('faithfulness', 0),
        'tone_compliance': s.get('tone_compliance', 0),
        'boundary_correctness': s.get('boundary_correctness', 0),
        'must_not_violation': s.get('must_not_violation', False),
        'is_hallucination': s.get('is_hallucination', False),
        'contains_expected_points': s.get('contains_expected_points', False),
        'comment': s.get('comment', ''),
    })
df = pd.DataFrame(rows)

print("=" * 60)
print("📊 评测报告 v4.0")
print("=" * 60)
print(f"总用例数：{total}")

intent_ok = (df['intent_accuracy'] >= 4).sum()
branch_ok = (df['branch_routing'] >= 4).sum()
hal_cnt = df['is_hallucination'].sum()
vio_cnt = df['must_not_violation'].sum()
pts_ok = df['contains_expected_points'].sum()

print(f"\n--- 准确率 ---")
print(f"意图识别: {intent_ok}/{total} = {intent_ok/total*100:.1f}%")
print(f"分支路由: {branch_ok}/{total} = {branch_ok/total*100:.1f}%")
print(f"要点覆盖: {pts_ok}/{total} = {pts_ok/total*100:.1f}%")
print(f"\n--- 问题检测 ---")
print(f"🚨 幻觉率: {hal_cnt/total*100:.1f}%")
print(f"⚠️ 禁止表述违规率: {vio_cnt/total*100:.1f}%")

print(f"\n--- 平均分（0-5）---")
for col, name in [('intent_accuracy','意图识别'), ('branch_routing','分支路由'),
                   ('answer_relevance','答案相关性'), ('information_completeness','信息完整性'),
                   ('faithfulness','忠实度'), ('tone_compliance','语气合规'),
                   ('boundary_correctness','边界正确性')]:
    print(f"  {name}: {df[col].mean():.2f}")

print(f"\n--- 各分支 ---")
for cat in df['category'].unique():
    cdf = df[df['category'] == cat]
    ok = (cdf['branch_routing'] >= 4).sum()
    print(f"  {cat:8s}: {ok}/{len(cdf)} = {ok/len(cdf)*100:.1f}% | 均分{cdf['branch_routing'].mean():.2f}")

print(f"\n--- 按测试类型 ---")
for tt in df['test_type'].unique():
    tdf = df[df['test_type'] == tt]
    ok = (tdf['branch_routing'] >= 4).sum()
    print(f"  {tt:15s}: {ok}/{len(tdf)} = {ok/len(tdf)*100:.1f}%")

print(f"\n--- 按难度 ---")
for d in ['简单', '中等', '困难']:
    ddf = df[df['difficulty'] == d]
    if len(ddf) > 0:
        ok = (ddf['branch_routing'] >= 4).sum()
        print(f"  {d}: {ok}/{len(ddf)} = {ok/len(ddf)*100:.1f}%")

# 导出
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(OUTPUT_DIR, f"eval_results_{ts}.csv")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
json_path = os.path.join(OUTPUT_DIR, f"eval_results_{ts}.json")
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\n💾 CSV: {csv_path}")
print(f"💾 JSON: {json_path}")

📊 评测报告 v4.0
总用例数：505

--- 准确率 ---
意图识别: 473/505 = 93.7%
分支路由: 462/505 = 91.5%
要点覆盖: 440/505 = 87.1%

--- 问题检测 ---
🚨 幻觉率: 2.6%
⚠️ 禁止表述违规率: 0.8%

--- 平均分（0-5）---
  意图识别: 4.81
  分支路由: 4.70
  答案相关性: 4.70
  信息完整性: 4.32
  忠实度: 4.85
  语气合规: 4.93
  边界正确性: 4.82

--- 各分支 ---
  商品咨询    : 268/281 = 95.4% | 均分4.83
  订单查询    : 36/36 = 100.0% | 均分5.00
  退款换货    : 61/70 = 87.1% | 均分4.56
  投诉      : 37/47 = 78.7% | 均分4.23
  其他      : 46/49 = 93.9% | 均分4.69
  边界混淆    : 14/22 = 63.6% | 均分3.95

--- 按测试类型 ---
  single_turn    : 362/382 = 94.8%
  multi_turn     : 14/17 = 82.4%
  boundary       : 31/35 = 88.6%
  prohibited     : 17/23 = 73.9%
  adversarial    : 38/48 = 79.2%

--- 按难度 ---
  简单: 184/190 = 96.8%
  中等: 158/182 = 86.8%
  困难: 120/133 = 90.2%

💾 CSV: eval_output\eval_results_20260807_142126.csv
💾 JSON: eval_output\eval_results_20260807_142126.json


## Cell 7: 失败用例分析

In [10]:
failed = df[df['branch_routing'] < 4]
print(f"失败用例: {len(failed)}/{len(df)}")
if len(failed) > 0:
    for _, row in failed.head(30).iterrows():
        print(f"\n[{row['case_id']}] {row['category']} | {row['test_type']} | {row['difficulty']}")
        print(f"  分支={row['branch_routing']}/5 幻觉={row['is_hallucination']} 违规={row['must_not_violation']}")
        print(f"  评语: {row['comment']}")
    if len(failed) > 30:
        print(f"\n... 还有 {len(failed)-30} 条")
else:
    print("🎉 无失败用例！")

print(f"\n--- 各分类失败率 ---")
for cat in df['category'].unique():
    cdf = df[df['category'] == cat]
    fr = (cdf['branch_routing'] < 4).mean() * 100
    print(f"  {cat}: {fr:.1f}% ({len(cdf)}条)")

失败用例: 43/505

[TC-IF6-IF] 投诉 | single_turn | 中等
  分支=2/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未体现投诉分支核心动作（致歉、补偿），仅机械索要订单号，缺失期望要点‘抱歉’和‘补偿’。

[TC-BOUND-010] 商品咨询 | boundary | 困难
  分支=1/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未响应边界值（4999<5000），未给出‘不需要本人签收’等关键要点，错误推诿至知识库缺失。

[TC-MULTI-002] 商品咨询 | multi_turn | 困难
  分支=1/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未回应已明确的商品（WF-1000XM5）和问题（续航），且未提供已知关键信息‘30小时’，属严重信息缺失。

[TC-PROHIB-001] 退款换货 | prohibited | 中等
  分支=1/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。路由错误：用户问退款到账时效，属'退款咨询'分支，非'商品咨询'分支。

[TC-PROHIB-003] 商品咨询 | prohibited | 中等
  分支=3/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未包含期望要点'48小时'和'工作日'，且将换货时效答为3-7个工作日，偏离用户询问的'多久能收到'核心诉求。

[TC-PROHIB-005] 商品咨询 | prohibited | 中等
  分支=1/5 幻觉=True 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未提及'16:00/当日/次日'任一要点；虚构'节假日发货安排''双十一时效延长'等无依据信息；路由错误，应答商品咨询却转向发货政策泛化说明。

[TC-ADV-001] 边界混淆 | adversarial | 困难
  分支=1/5 幻觉=False 违规=False
  评语: [预检修正]must_not未违规(精确匹配)。未体现‘抱歉、订

## Cell 8: 清理断点（需要重新评测时运行）

In [40]:
# 清理断点文件，从头开始评测
ckpt_path = os.path.join(OUTPUT_DIR, "checkpoint.json")
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print(f"✅ 断点已清理: {ckpt_path}")
    print("现在可以重新运行 Cell 5 从头评测")
else:
    print("无断点文件")

无断点文件


cell 9:清理指定用例

In [66]:
import json
import os

# checkpoint 文件路径（和 Cell 5 里 OUTPUT_DIR 一致）
checkpoint_path = "eval_output/checkpoint.json"

# 要删除的 7 条失败用例 ID
ids_to_remove = [
   "TC-IF6-IF",       # 投诉：未体现致歉+补偿
    "TC-BOUND-010",    # 商品咨询：4999<5000 边界未响应
    "TC-MULTI-002",    # 商品咨询：多轮上下文未继承商品名
    "TC-PROHIB-001",   # 退款换货：退款时效路由错+幻觉
    "TC-PROHIB-003",   # 商品咨询：换货48小时未答
    "TC-PROHIB-005",   # 商品咨询：发货时效虚构+要点遗漏
    "TC-ADV-001", 
]

# 读取
with open(checkpoint_path, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"删除前：{len(results)} 条")

# 过滤掉要删除的
before = len(results)
results = [r for r in results if r["case_id"] not in ids_to_remove]
after = len(results)

# 保存回去
with open(checkpoint_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"删除了：{before - after} 条")
print(f"删除后：{after} 条")
print(f"剩余 case_id 示例：{[r['case_id'] for r in results[:5]]}")

删除前：102 条
删除了：7 条
删除后：95 条
剩余 case_id 示例：['TC-INTENT-001', 'TC-INTENT-002', 'TC-INTENT-003', 'TC-INTENT-004', 'TC-INTENT-005']
